In [60]:
chain = "AB"
healthy = [f"/dsi/efroni-lab/sbm/chen2/raw_data/healthy/{chain}_h_v1", f"/dsi/efroni-lab/sbm/chen2/raw_data/healthy/{chain}_h_v2"]
# sick = f"/dsi/efroni-lab/sbm/chen2/raw_data/sick/mixcr_4.7_{chain}"
sick = f"/dsi/efroni-lab/sbm/chen2/raw_data/ovarian/{chain}_oc"

downsample_path = f"/dsi/efroni-lab/sbm/chen2/downsampled_data/ovarian/{chain}"

In [61]:
import numpy as np
import pandas as pd
import chephy_model as cpm

def downsample_mixcr_depth(df: pd.DataFrame, n: int = 10000, count_col: str = "readCount") -> pd.DataFrame:
    if count_col not in df.columns:
        raise ValueError(f"{count_col} not found in DataFrame.")

    counts = pd.to_numeric(df[count_col], errors="coerce").to_numpy(dtype=float)
    total = np.nansum(counts)

    if not np.isfinite(total) or total <= 0:
        raise ValueError("Total readCount must be > 0.")
    if n > total:
        print("skipping downsampling as n > total reads")
        return
        # raise ValueError(f"Requested n={n} is larger than total reads={int(total)}.")

    scaled = counts * (n / total)
    base = np.floor(scaled).astype(int)
    remainder = n - base.sum()

    frac = scaled - base
    if remainder > 0:
        # stable sort so ties keep original order (deterministic)
        idx = np.argsort(-frac, kind="mergesort")
        base[idx[:remainder]] += 1

    out = df.copy()
    out[count_col] = base
    out = out[out[count_col] > 0].copy()

    if int(out[count_col].sum()) != n:
        raise RuntimeError("Downsampling failed to reach exact target depth.")
    return out

def top_k_by_readcount(df, k, count_col="readCount"):
    if count_col not in df.columns:
        raise ValueError(f"{count_col} not found")

    return (
        df
        .sort_values(by=count_col, ascending=False)
        .head(k)
        .copy()
    )

def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered


def prepare_and_truncate(df, cdr3_header):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    return df

In [62]:
from glob import glob
import os

cdr3_header = 'aaSeqCDR3' 
downsample_size = 20000
top_k = 600
output_dir = f"{downsample_path}/{top_k}"
os.makedirs(output_dir, exist_ok=True)

def load_patient_files(paths, label):
    for path in paths if isinstance(paths, list) else [paths]:
        files = glob(f"{path}/*.tsv")
        
        for f in files:
            # print("Processing file:", f)
            df = pd.read_csv(f, sep='\t', low_memory=False)
            # if "CAVRDSNYQLIW" in list(df["aaSeqCDR3"]):
            #     print("in original", "file:", f)

            df_ds = downsample_mixcr_depth(df, n=downsample_size, count_col="readCount")
            if df_ds is None:
                continue

            df_top = top_k_by_readcount(df_ds, k=top_k, count_col="readCount")
            # if "CAVRDSNYQLIW" in list(df_top["aaSeqCDR3"]):
            #     print("in top_k", "file:", f)

            # print("before:", df.shape, "after:", df_ds.shape)
            pid = os.path.basename(f)
            out_path = os.path.join(output_dir, f"{label}_{pid}")
            df_top.to_csv(out_path, sep='\t', index=False)

# Replace with your folder paths
healthy_patients = load_patient_files(healthy, label="healthy")
sick_patients = load_patient_files(sick, label="sick")

skipping downsampling as n > total reads
